<a href="https://colab.research.google.com/github/Pooja-sethi12/ABC-Ltd-churn/blob/main/model.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# ==========================================
# STEP 1: IMPORT LIBRARIES & SETUP
# ==========================================
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import joblib

from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression, LogisticRegression
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score, roc_auc_score,
    mean_squared_error, r2_score, classification_report, confusion_matrix
)
from sklearn.preprocessing import StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.pipeline import Pipeline

# Set seed for reproducibility
np.random.seed(42)

print("Libraries imported successfully.\n")


# ==========================================
# STEP 2: GENERATE DATASET FOR ABC LTD.
# ==========================================
# Simulating a Telco/Subscription Churn dataset for ABC Ltd.
n_samples = 2000

data = pd.DataFrame({
    'Tenure_Months': np.random.randint(1, 72, size=n_samples),
    'Monthly_Charges': np.random.uniform(20.0, 120.0, size=n_samples),
    'Contract_Type': np.random.choice(['Month-to-month', 'One year', 'Two year'], size=n_samples, p=[0.55, 0.25, 0.20]),
    'Payment_Method': np.random.choice(['Electronic check', 'Mailed check', 'Bank transfer', 'Credit card'], size=n_samples),
    'Paperless_Billing': np.random.choice(['Yes', 'No'], size=n_samples, p=[0.6, 0.4]),
    'Tech_Support': np.random.choice(['Yes', 'No'], size=n_samples, p=[0.35, 0.65])
})

# Create log-odds formula to assign realistic churn probabilities
log_odds = (
    - 0.05 * data['Tenure_Months']
    + 0.02 * data['Monthly_Charges']
    + (data['Contract_Type'] == 'Month-to-month') * 1.2
    - (data['Contract_Type'] == 'Two year') * 1.0
    + (data['Payment_Method'] == 'Electronic check') * 0.5
    - (data['Tech_Support'] == 'Yes') * 0.6
    - 0.5
)
probabilities = 1 / (1 + np.exp(-log_odds))
data['Churn'] = (np.random.rand(n_samples) < probabilities).astype(int)

print(f"Dataset generated for ABC Ltd. Total Rows: {data.shape[0]}, Columns: {data.shape[1]}")
print(f"Churn Distribution:\n{data['Churn'].value_counts(normalize=True)}\n")


# ==========================================
# STEP 3: PREPROCESS & TRAIN-TEST SPLIT
# ==========================================
X = data.drop(columns=['Churn'])
y = data['Churn']

# Define feature types
num_features = ['Tenure_Months', 'Monthly_Charges']
cat_features = ['Contract_Type', 'Payment_Method', 'Paperless_Billing', 'Tech_Support']

# Define preprocessor
preprocessor = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), num_features),
        ('cat', OneHotEncoder(drop='first', sparse_output=False), cat_features)
    ]
)

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

# Fit preprocessor on training data
X_train_prep = preprocessor.fit_transform(X_train)
X_test_prep = preprocessor.transform(X_test)

# Get feature names after encoding
cat_encoder = preprocessor.named_transformers_['cat']
encoded_cat_names = cat_encoder.get_feature_names_out(cat_features)
feature_names = num_features + list(encoded_cat_names)


# ==========================================
# STEP 4: MODEL 1 — LINEAR REGRESSION
# ==========================================
lin_model = LinearRegression()
lin_model.fit(X_train_prep, y_train)

y_pred_lin = lin_model.predict(X_test_prep)
# Convert continuous predictions to binary classification (Threshold = 0.5)
y_pred_lin_binary = (y_pred_lin >= 0.5).astype(int)

# Evaluation metrics
lin_mse = mean_squared_error(y_test, y_pred_lin)
lin_acc = accuracy_score(y_test, y_pred_lin_binary)
print("=== MODEL 1: LINEAR REGRESSION BENCHMARK ===")
print(f"Mean Squared Error (MSE): {lin_mse:.4f}")
print(f"Accuracy (Threshold 0.5): {lin_acc:.4f}")
print(f"Min Prediction Value: {y_pred_lin.min():.4f} (Notice out-of-bounds < 0 values)")
print(f"Max Prediction Value: {y_pred_lin.max():.4f} (Notice out-of-bounds > 1 values)\n")


# ==========================================
# STEP 5: MODEL 2 — LOGISTIC REGRESSION
# ==========================================
log_model = LogisticRegression(max_iter=1000)
log_model.fit(X_train_prep, y_train)

y_pred_log = log_model.predict(X_test_prep)
y_prob_log = log_model.predict_proba(X_test_prep)[:, 1]

# Metrics
log_acc = accuracy_score(y_test, y_pred_log)
log_prec = precision_score(y_test, y_pred_log)
log_rec = recall_score(y_test, y_pred_log)
log_f1 = f1_score(y_test, y_pred_log)
log_auc = roc_auc_score(y_test, y_prob_log)

print("=== MODEL 2: LOGISTIC REGRESSION (PRIMARY) ===")
print(f"Accuracy:  {log_acc:.4f}")
print(f"Precision: {log_prec:.4f}")
print(f"Recall:    {log_rec:.4f}")
print(f"F1 Score:  {log_f1:.4f}")
print(f"ROC-AUC:   {log_auc:.4f}\n")


# ==========================================
# STEP 6: EXPLAINABILITY (ODDS RATIOS)
# ==========================================
coefficients = log_model.coef_[0]
odds_ratios = np.exp(coefficients)

interpretability_df = pd.DataFrame({
    'Feature': feature_names,
    'Coefficient': coefficients,
    'Odds_Ratio': odds_ratios
}).sort_values(by='Odds_Ratio', ascending=False)

print("=== FEATURE IMPORTANCE & ODDS RATIOS ===")
print(interpretability_df.to_string(index=False))

print("\nInterpretation Note for Assignment:")
print("- Odds Ratio > 1.0 means increasing this feature INCREASES churn risk.")
print("- Odds Ratio < 1.0 means increasing this feature DECREASES churn risk (protective factor).\n")


# ==========================================
# STEP 7: BUILD & EXPORT STREAMLIT PIPELINE
# ==========================================
full_pipeline = Pipeline([
    ('preprocessor', preprocessor),
    ('model', log_model)
])

# Fit full pipeline on entire dataset
full_pipeline.fit(X, y)

# Save pipeline object to file
joblib.dump(full_pipeline, 'model.pkl')
print("✅ Success! 'model.pkl' has been exported and is ready to download for your Streamlit app.")

Libraries imported successfully.

Dataset generated for ABC Ltd. Total Rows: 2000, Columns: 7
Churn Distribution:
Churn
0    0.5815
1    0.4185
Name: proportion, dtype: float64

=== MODEL 1: LINEAR REGRESSION BENCHMARK ===
Mean Squared Error (MSE): 0.1843
Accuracy (Threshold 0.5): 0.7375
Min Prediction Value: -0.3181 (Notice out-of-bounds < 0 values)
Max Prediction Value: 1.0607 (Notice out-of-bounds > 1 values)

=== MODEL 2: LOGISTIC REGRESSION (PRIMARY) ===
Accuracy:  0.7425
Precision: 0.7078
Recall:    0.6527
F1 Score:  0.6791
ROC-AUC:   0.8009

=== FEATURE IMPORTANCE & ODDS RATIOS ===
                        Feature  Coefficient  Odds_Ratio
                Monthly_Charges     0.660370    1.935508
Payment_Method_Electronic check     0.611893    1.843918
    Payment_Method_Mailed check     0.430949    1.538716
     Payment_Method_Credit card     0.423154    1.526770
          Paperless_Billing_Yes     0.207479    1.230571
               Tech_Support_Yes    -0.881244    0.414267
     